In [1]:
import json
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from medical_rag.chroma_indexer import PubMedChromaIndexer

e:\anaconda3\envs\medrag\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 初始化并构建索引

In [2]:
chunks_df = pd.read_parquet(
    "F:\RAG\data\pubmed_fulltext_chunks_2.parquet"
)

In [3]:
#初始化索引器

indexer = PubMedChromaIndexer(
    model_name="BAAI/bge-small-en-v1.5",
    persist_directory=r"F:\RAG\vector_db\chroma_pubmed_db_rebuilt",
    collection_name="pubmed_fulltext_bge_small",
    device="cuda",
    embedding_batch_size=32,
    insert_batch_size=1000,
    load_existing=True,
)

Loading embedding model: BAAI/bge-small-en-v1.5
Device: cuda
Persist directory: F:\RAG\vector_db\chroma_pubmed_db_rebuilt
Collection name: pubmed_fulltext_bge_small


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3322.52it/s]
F:\RAG\src\medical_rag\chroma_indexer.py:109: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  dimension = self.embedding_model.get_sentence_embedding_dimension()


Existing collections: ['pubmed_fulltext_bge_small']
Collection loaded successfully. Count: 631,000
Embedding dimension: 384
Distance metric: cosine


In [ ]:
stats = indexer.build_index(
    chunks_df=chunks_df,
    stats_path=(
        "F:/RAG/"
        "pubmed_fulltext_bge_small_index_stats.json"
    ),
    recreate_collection=True,
)

Valid chunks to index: 668,461
Embedding dimension: 384


Embedding and indexing:  94%|█████████▍| 631/669 [1:48:23<06:31, 10.31s/it]


InternalError: Error in compaction: Failed to apply logs to the metadata segment

## 单独生成文档和查询向量

In [4]:
#生成文档向量

sample_documents = chunks_df["text"].head(8).tolist()

document_embeddings = indexer.encode_documents(
    sample_documents
)

print(document_embeddings.shape)
print(document_embeddings.dtype)

(8, 384)
float32


In [5]:
#生成查询向量

query_embedding = indexer.encode_query(
    "What are the clinical risk factors for acute kidney injury?"
)

print(query_embedding.shape)

(1, 384)


## 进行普通查询

In [6]:
results_df = indexer.query(
    query_text=(
        "What biomarkers are associated with "
        "acute kidney injury in critically ill patients?"
    ),
    n_results=5,
)

results_df[
    [
        "rank",
        "similarity",
        "source_title",
        "journal",
        "publication_year",
        "pmid",
        "text",
    ]
]

,rank,similarity,source_title,journal,publication_year,pmid,text
0,1,0.810452,Differentially expressed miRNAs in sepsis-indu...,PLoS ONE,2017,28296904,Title: Differentially expressed miRNAs in seps...
1,2,0.797292,ESICM LIVES 2016: part three,Intensive Care Medicine Experimental,2016,,Title: ESICM LIVES 2016: part three\n\n. Concl...
2,3,0.794136,Differentially expressed miRNAs in sepsis-indu...,PLoS ONE,2017,28296904,Title: Differentially expressed miRNAs in seps...
3,4,0.793928,Risk assessment of acute kidney injury followi...,Journal of Cardiothoracic Surgery,2021,33407652,Title: Risk assessment of acute kidney injury ...
4,5,0.790957,Differentially expressed miRNAs in sepsis-indu...,PLoS ONE,2017,28296904,Title: Differentially expressed miRNAs in seps...


In [7]:
#打印查询完整结果

for _, row in results_df.iterrows():
    print("=" * 100)
    print("Rank:", row["rank"])
    print("Similarity:", f"{row['similarity']:.4f}")
    print("Title:", row["source_title"])
    print("Journal:", row["journal"])
    print("Year:", row["publication_year"])
    print("PMID:", row["pmid"])
    print("Text:")
    print(row["text"][:1000])

Rank: 1
Similarity: 0.8105
Title: Differentially expressed miRNAs in sepsis-induced acute kidney injury target oxidative stress and mitochondrial dysfunction pathways
Journal: PLoS ONE
Year: 2017
PMID: 28296904
Text:
Title: Differentially expressed miRNAs in sepsis-induced acute kidney injury target oxidative stress and mitochondrial dysfunction pathways

Despite advances in the treatment of critically ill patients, the development of acute kidney injury (AKI) shows a high mortality rate [1]. The mortality rate was 34% in patients with AKI versus 7% in patients without AKI [2]. The cause of AKI in critically ill patients is often multifactorial, while sepsis is the most common cause which is up to 50% [3, 4]. And sepsis-induced AKI is diagnosed in up to 47.9% of ICU patients, it is associated with increased progression to chronic kidney disease (CKD) [1]. Therefore sepsis-induced AKI should be intervened as early as possible. But the mechanisms underlying this event are not fully under

## Metadata Filter 查询

In [8]:
#指定期刊

nature_results = indexer.query(
    query_text=(
        "machine learning methods for disease prediction"
    ),
    n_results=10,
    where_filter={
        "journal": "Nature Communications"
    },
)

In [9]:
nature_results[
    [
        "rank",
        "source_title",
        "journal",
        "publication_year",
        "similarity",
    ]
]


,rank,source_title,journal,publication_year,similarity
0,1,On the performance of pre-microRNA detection a...,Nature Communications,2017,0.768073
1,2,On the performance of pre-microRNA detection a...,Nature Communications,2017,0.747758
2,3,On the performance of pre-microRNA detection a...,Nature Communications,2017,0.746543
3,4,On the performance of pre-microRNA detection a...,Nature Communications,2017,0.738841
4,5,On the performance of pre-microRNA detection a...,Nature Communications,2017,0.737416
5,6,On the performance of pre-microRNA detection a...,Nature Communications,2017,0.731410
6,7,On the performance of pre-microRNA detection a...,Nature Communications,2017,0.724651
7,8,On the performance of pre-microRNA detection a...,Nature Communications,2017,0.719163
8,9,A Bayesian mixture model for clustering drople...,Nature Communications,2019,0.715835
9,10,Decoding individual differences in STEM learni...,Nature Communications,2019,0.715370


In [10]:
# 近五年 Nature Communications 文献
latest_year = int(
    chunks_df["pub_date"]
    .astype(str)
    .str.extract(r"((?:19|20)\d{2})")[0]
    .dropna()
    .astype(int)
    .max()
)

start_year = latest_year - 4

recent_nature_results = indexer.query(
    query_text=(
        "biomedical machine learning and clinical prediction"
    ),
    n_results=10,
    where_filter={
        "$and": [
            {
                "journal": "Nature Communications"
            },
            {
                "publication_year": {
                    "$gte": start_year
                }
            }
        ]
    },
)

print(
    f"Filtering publications from "
    f"{start_year} to {latest_year}"
)

recent_nature_results[
    [
        "rank",
        "source_title",
        "journal",
        "publication_year",
        "similarity",
    ]
]

Filtering publications from 2022 to 2026


,rank,source_title,journal,publication_year,similarity
0,1,KSTAR: An algorithm to predict patient-specifi...,Nature Communications,2022,0.704817
1,2,KSTAR: An algorithm to predict patient-specifi...,Nature Communications,2022,0.704132
2,3,KSTAR: An algorithm to predict patient-specifi...,Nature Communications,2022,0.702878
3,4,Cutaneous and acral melanoma cross-OMICs revea...,Nature Communications,2022,0.701875
4,5,Automated identification of sequence-tailored ...,Nature Communications,2022,0.700966
5,6,Cutaneous and acral melanoma cross-OMICs revea...,Nature Communications,2022,0.700164
6,7,Cutaneous and acral melanoma cross-OMICs revea...,Nature Communications,2022,0.700129
7,8,KSTAR: An algorithm to predict patient-specifi...,Nature Communications,2022,0.696720
8,9,KSTAR: An algorithm to predict patient-specifi...,Nature Communications,2022,0.696540
9,10,Automated identification of sequence-tailored ...,Nature Communications,2022,0.696069


## 质量验证

In [ ]:
#基础统计验证
#向量数据库里包含index的向量有631,000个

collection_count = indexer.collection.count()
expected_count = len(
    chunks_df[
        chunks_df["text"]
        .fillna("")
        .astype(str)
        .str.strip()
        .ne("")
    ]
)

basic_validation = pd.DataFrame([
    {
        "metric": "Expected vectors",
        "value": expected_count,
        "status": "INFO",
    },
    {
        "metric": "Indexed vectors",
        "value": collection_count,
        "status": (
            "PASS"
            if collection_count == expected_count
            else "FAIL"
        ),
    },
    {
        "metric": "Embedding dimension",
        "value": indexer.embedding_dimension,
        "status": "PASS",
    },
])

basic_validation

,metric,value,status
0,Expected vectors,668461,INFO
1,Indexed vectors,631000,FAIL
2,Embedding dimension,384,PASS


In [12]:
#样本元数据验证

sample_records = indexer.collection.peek(
    limit=5
)

for i in range(len(sample_records["ids"])):
    print("=" * 100)
    print("ID:", sample_records["ids"][i])
    print("Metadata:", sample_records["metadatas"][i])
    print("Document:")
    print(sample_records["documents"][i][:500])

ID: PMID_30731006_chunk_0000
Metadata: {'total_chunks': 17, 'publication_year': 2019, 'pub_date': '2019', 'journal': 'PLoS ONE', 'token_count': 500, 'split_strategy': 'sliding_window', 'pmid': '30731006', 'chunk_index': 0, 'doc_id': 'PMID_30731006', 'source_title': 'Common mental disorders and subjective well-being: Emotional training among medical students based on positive psychology'}
Document:
Title: Common mental disorders and subjective well-being: Emotional training among medical students based on positive psychology

In the field of positive psychology, well-being is divided into eudaimonic well-being (EWB) and subjective well-being (SWB). EWB, which is also called psychological well-being, is linked to the realization of intimate potential, and consists of some parameters such as positive relationships and self-acceptance. SWB, which is also called hedonic well-being, is connecte
ID: PMID_30731006_chunk_0001
Metadata: {'pmid': '30731006', 'pub_date': '2019', 'journal': 'PLoS O

In [15]:
# 自相似性检索验证
# 文本片段检索：从索引中抽取一条正文，把正文片段作为查询，期望原Chunk排在前列

def validate_self_similarity(
    indexer: PubMedChromaIndexer,
    sample_size: int = 20,
    n_results: int = 5,
    random_state: int = 42,
) -> pd.DataFrame:

    rng = np.random.default_rng(random_state)

    total_count = indexer.collection.count()

    offsets = rng.choice(
        total_count,
        size=min(sample_size, total_count),
        replace=False,
    )

    validation_rows = []

    for offset in tqdm(
        offsets,
        desc="Self-similarity validation"
    ):
        record = indexer.collection.get(
            limit=1,
            offset=int(offset),
            include=["documents", "metadatas"],
        )

        source_id = record["ids"][0]
        source_text = record["documents"][0]

        # 使用一段正文而不是整块，模拟真实查询
        query_text = source_text[:1000]

        result_df = indexer.query(
            query_text=query_text,
            n_results=n_results,
        )

        retrieved_ids = result_df["vector_id"].tolist()

        if source_id in retrieved_ids:
            rank = retrieved_ids.index(source_id) + 1
            hit = True
        else:
            rank = None
            hit = False

        validation_rows.append({
            "source_vector_id": source_id,
            "hit_in_top_k": hit,
            "retrieved_rank": rank,
            "top_result_id": (
                retrieved_ids[0]
                if retrieved_ids
                else None
            ),
            "top_similarity": (
                result_df.iloc[0]["similarity"]
                if not result_df.empty
                else None
            ),
        })

    return pd.DataFrame(validation_rows)

In [ ]:
self_similarity_df = validate_self_similarity(
    indexer=indexer,
    sample_size=20,
    n_results=5,
)

print(
    "Top-5 self-similarity hit rate:",
    f"{self_similarity_df['hit_in_top_k'].mean() * 100:.2f}%"
)

self_similarity_df

Self-similarity validation: 100%|██████████| 20/20 [00:29<00:00,  1.48s/it]

Top-5 self-similarity hit rate: 90.00%


,source_vector_id,hit_in_top_k,retrieved_rank,top_result_id,top_similarity
0,PMID_34689825_chunk_0037,True,1.0,PMID_34689825_chunk_0037,0.959098
1,PMID_31692805_chunk_0016,True,1.0,PMID_31692805_chunk_0016,0.944172
2,PMID_32607151_chunk_0020,True,1.0,PMID_32607151_chunk_0020,0.963435
3,DOC_c688a3ed7456_chunk_0002,True,1.0,DOC_c688a3ed7456_chunk_0002,0.966016
4,PMID_36246146_chunk_0009,True,1.0,PMID_36246146_chunk_0009,0.940000
5,DOC_db293fd5304f_chunk_0010,True,1.0,DOC_db293fd5304f_chunk_0010,0.977970
6,PMID_24914639_chunk_0006,True,1.0,PMID_24914639_chunk_0006,0.954180
7,PMID_31885491_chunk_0216,False,NaN,PMID_31885491_chunk_0192,0.967343
8,PMID_22918376_chunk_0007,True,1.0,PMID_22918376_chunk_0007,0.967275
9,PMID_35585532_chunk_0048,True,1.0,PMID_35585532_chunk_0048,0.975909


In [16]:
# 原向量自匹配
# 直接从数据库取出一个向量，再用该向量查询

def validate_exact_vector_self_match(
    indexer: PubMedChromaIndexer,
    limit: int = 10,
) -> pd.DataFrame:

    records = indexer.collection.get(
        limit=limit,
        include=[
            "documents",
            "metadatas",
            "embeddings",
        ],
    )

    rows = []

    for i, source_id in enumerate(records["ids"]):
        source_embedding = records["embeddings"][i]

        result = indexer.collection.query(
            query_embeddings=[source_embedding],
            n_results=1,
            include=["distances"],
        )

        top_id = result["ids"][0][0]
        distance = result["distances"][0][0]

        rows.append({
            "source_id": source_id,
            "top_id": top_id,
            "distance": distance,
            "exact_self_match": source_id == top_id,
        })

    return pd.DataFrame(rows)

In [17]:
exact_match_df = validate_exact_vector_self_match(
    indexer,
    limit=20,
)

print(
    "Exact vector self-match rate:",
    exact_match_df["exact_self_match"].mean() * 100
)

Exact vector self-match rate: 100.0


## 边界情况验证

In [18]:
#空查询

try:
    indexer.query(
        query_text="",
        n_results=5
    )
except ValueError as error:
    print("PASS:", error)

PASS: query_text cannot be empty.


In [19]:
#只有空格

try:
    indexer.query(
        query_text="     ",
        n_results=5
    )
except ValueError as error:
    print("PASS:", error)

PASS: query_text cannot be empty.


In [20]:
#超长查询

long_query = (
    "acute kidney injury biomarker prediction "
    * 1000
)

safe_query = indexer.truncate_text_by_tokens(
    long_query,
    max_tokens=512
)

print(
    "Original query tokens:",
    len(
        indexer.embedding_model.tokenizer.encode(
            long_query,
            add_special_tokens=False
        )
    )
)

print(
    "Truncated query tokens:",
    len(
        indexer.embedding_model.tokenizer.encode(
            safe_query,
            add_special_tokens=False
        )
    )
)

long_query_results = indexer.query(
    query_text=safe_query,
    n_results=5,
)

long_query_results[
    [
        "rank",
        "source_title",
        "similarity",
    ]
]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (7000 > 512). Running this sequence through the model will result in indexing errors


Original query tokens: 7000
Truncated query tokens: 510


,rank,source_title,similarity
0,1,Risk assessment of acute kidney injury followi...,0.775715
1,2,Incidence and Prognosis of Acute Kidney Diseas...,0.773047
2,3,Intra-individual variability of eGFR trajector...,0.770514
3,4,ESICM LIVES 2016: part three,0.769685
4,5,NephroCheck: should we consider urine osmolality?,0.769602


## 质量验证报告

In [ ]:
metadata_sample = indexer.collection.peek(
    limit=1
)["metadatas"][0]

validation_stats = {
    "validated_at": pd.Timestamp.now().isoformat(),
    "collection_name": indexer.collection_name,
    "expected_vector_count": int(expected_count),
    "actual_vector_count": int(
        indexer.collection.count()
    ),
    "vector_count_match": bool(
        indexer.collection.count()
        == expected_count
    ),
    "embedding_model": indexer.model_name,
    "embedding_dimension": int(
        indexer.embedding_dimension
    ),
    "self_similarity_top5_hit_rate": float(
        self_similarity_df["hit_in_top_k"].mean()
    ),
    "exact_vector_self_match_rate": float(
        exact_match_df["exact_self_match"].mean()
    ),
    "sample_metadata": metadata_sample,
    "empty_query_handling": "raises ValueError",
    "long_query_handling": (
        "truncate to 512 tokens before embedding"
    ),
}

with open(
    "F:/RAG/data/vector_index_validation_stats.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        validation_stats,
        f,
        indent=4,
        ensure_ascii=False,
    )

print(json.dumps(
    validation_stats,
    indent=2,
    ensure_ascii=False
))

self_similarity_df.to_csv(
    "F:/RAG/data/self_similarity_validation.csv",
    index=False
)

recent_nature_results.to_csv(
    "F:/RAG/data/metadata_filter_test.csv",
    index=False
)

{
  "validated_at": "2026-08-19T19:21:11.032608",
  "collection_name": "pubmed_fulltext_bge_small",
  "expected_vector_count": 668461,
  "actual_vector_count": 631000,
  "vector_count_match": false,
  "embedding_model": "BAAI/bge-small-en-v1.5",
  "embedding_dimension": 384,
  "self_similarity_top5_hit_rate": 0.9,
  "exact_vector_self_match_rate": 1.0,
  "sample_metadata": {
    "split_strategy": "sliding_window",
    "pub_date": "2019",
    "source_title": "Common mental disorders and subjective well-being: Emotional training among medical students based on positive psychology",
    "journal": "PLoS ONE",
    "doc_id": "PMID_30731006",
    "token_count": 500,
    "chunk_index": 0,
    "publication_year": 2019,
    "total_chunks": 17,
    "pmid": "30731006"
  },
  "empty_query_handling": "raises ValueError",
  "long_query_handling": "truncate to 512 tokens before embedding"
}
